In [3]:
# =====================================================
# HR ANALYTICS — DATA CLEANING & EDA
# =====================================================

# Core Libraries
import pandas as pd
import numpy as np

# Visualization Libraries
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

# Statistical Libraries
from scipy import stats

# Utility Libraries
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

# Notebook Settings
%matplotlib inline

# Visualization Style
plt.style.use('ggplot')

sns.set_theme(
    style='whitegrid',
    palette='deep'
)

# Pandas Display Settings
pd.set_option('display.max_columns', None)

pd.set_option('display.max_rows', 100)

pd.set_option('display.float_format', '{:.2f}'.format)

print("Libraries loaded successfully")

Libraries loaded successfully


In [4]:
# =====================================================
# LOAD DATASETS
# =====================================================

BASE_PATH = Path.cwd().parent

RAW_DATA_PATH = BASE_PATH / "data" / "raw"

# Load datasets
attrition_df = pd.read_csv(
    RAW_DATA_PATH / "attrition_raw.csv"
)

recruitment_df = pd.read_csv(
    RAW_DATA_PATH / "recruitment_raw.csv"
)

# Standardize columns
attrition_df.columns = [
    col.lower().replace(" ", "_")
    for col in attrition_df.columns
]

recruitment_df.columns = [
    col.lower().replace(" ", "_")
    for col in recruitment_df.columns
]

print("Datasets loaded successfully")

print("\nAttrition Dataset Shape:")
print(attrition_df.shape)

print("\nRecruitment Dataset Shape:")
print(recruitment_df.shape)

Datasets loaded successfully

Attrition Dataset Shape:
(1470, 35)

Recruitment Dataset Shape:
(1500, 25)


In [5]:
# =====================================================
# DATA QUALITY AUDIT
# =====================================================

def data_audit(df, name):

    print("=" * 60)

    print(f"{name} DATASET AUDIT")

    print("=" * 60)

    print("\nShape:")
    print(df.shape)

    print("\nMissing Values:")
    print(df.isnull().sum()[df.isnull().sum() > 0])

    print("\nDuplicate Rows:")
    print(df.duplicated().sum())

    print("\nData Types:")
    print(df.dtypes)

    print("\nNumerical Summary:")
    print(df.describe())

# Run audits
data_audit(attrition_df, "ATTRITION")

print("\n\n")

data_audit(recruitment_df, "RECRUITMENT")

ATTRITION DATASET AUDIT

Shape:
(1470, 35)

Missing Values:
Series([], dtype: int64)

Duplicate Rows:
0

Data Types:
age                         int64
attrition                     str
businesstravel                str
dailyrate                   int64
department                    str
distancefromhome            int64
education                   int64
educationfield                str
employeecount               int64
employeenumber              int64
environmentsatisfaction     int64
gender                        str
hourlyrate                  int64
jobinvolvement              int64
joblevel                    int64
jobrole                       str
jobsatisfaction             int64
maritalstatus                 str
monthlyincome               int64
monthlyrate                 int64
numcompaniesworked          int64
over18                        str
overtime                      str
percentsalaryhike           int64
performancerating           int64
relationshipsatisfaction    int64

In [6]:
# =====================================================
# CREATE TARGET VARIABLE
# =====================================================

attrition_df['attrition_flag'] = np.where(
    attrition_df['attrition'] == 'Yes',
    1,
    0
)

print("Attrition target created")

print(
    attrition_df['attrition_flag']
    .value_counts()
)

Attrition target created
attrition_flag
0    1233
1     237
Name: count, dtype: int64


In [7]:
# =====================================================
# SALARY BAND FEATURE
# =====================================================

attrition_df['salary_band'] = pd.cut(

    attrition_df['monthlyincome'],

    bins=[0, 3000, 7000, 12000, 20000],

    labels=[
        'Low Income',
        'Middle Income',
        'High Income',
        'Executive'
    ]

)

print("Salary bands created")

print(
    attrition_df['salary_band']
    .value_counts()
)

Salary bands created
salary_band
Middle Income    640
Low Income       395
High Income      240
Executive        195
Name: count, dtype: int64


In [8]:
# =====================================================
# TENURE GROUP FEATURE
# =====================================================

attrition_df['tenure_group'] = pd.cut(

    attrition_df['yearsatcompany'],

    bins=[-1, 2, 5, 10, 40],

    labels=[
        'New Employee',
        'Junior Employee',
        'Mid-Level',
        'Veteran'
    ]

)

print("Tenure groups created")

print(
    attrition_df['tenure_group']
    .value_counts()
)

Tenure groups created
tenure_group
Mid-Level          448
Junior Employee    434
New Employee       342
Veteran            246
Name: count, dtype: int64


In [9]:
# =====================================================
# OVERTIME RISK FEATURE
# =====================================================

attrition_df['overtime_risk'] = np.where(

    (attrition_df['overtime'] == 'Yes') &
    (attrition_df['worklifebalance'] <= 2),

    'High Risk',

    'Normal'

)

print("Overtime risk feature created")

print(
    attrition_df['overtime_risk']
    .value_counts()
)

Overtime risk feature created
overtime_risk
Normal       1344
High Risk     126
Name: count, dtype: int64


In [10]:
# =====================================================
# EXPERIENCE CATEGORY
# =====================================================

recruitment_df['experience_category'] = pd.cut(

    recruitment_df['experience_years'],

    bins=[-1, 2, 5, 10, 20],

    labels=[
        'Fresher',
        'Junior',
        'Mid-Level',
        'Senior'
    ]

)

print("Experience category created")

print(
    recruitment_df['experience_category']
    .value_counts()
)

Experience category created
experience_category
Mid-Level    527
Fresher      446
Junior       344
Senior       183
Name: count, dtype: int64


In [11]:
# =====================================================
# HIRING SUCCESS FEATURE
# =====================================================

recruitment_df['hiring_success'] = np.where(

    recruitment_df['accepted_offer'] == 1,

    'Successful Hire',

    'Not Hired'

)

print("Hiring success feature created")

print(
    recruitment_df['hiring_success']
    .value_counts()
)

Hiring success feature created
hiring_success
Not Hired          1240
Successful Hire     260
Name: count, dtype: int64


In [12]:
# =====================================================
# REMOVE LOW VALUE COLUMNS
# =====================================================

drop_columns = [

    'employeecount',
    'standardhours',
    'over18'

]

attrition_df.drop(
    columns=drop_columns,
    inplace=True
)

print("Low value columns removed")

print(attrition_df.shape)

Low value columns removed
(1470, 36)


In [13]:
# =====================================================
# FINAL DATA VALIDATION
# =====================================================

print("=" * 50)

print("FINAL ATTRITION DATASET")

print("=" * 50)

print(attrition_df.head())

print("\n")

print(attrition_df.info())

print("\n")

print("=" * 50)

print("FINAL RECRUITMENT DATASET")

print("=" * 50)

print(recruitment_df.head())

print("\n")

print(recruitment_df.info())

FINAL ATTRITION DATASET
   age attrition     businesstravel  dailyrate              department  \
0   41       Yes      Travel_Rarely       1102                   Sales   
1   49        No  Travel_Frequently        279  Research & Development   
2   37       Yes      Travel_Rarely       1373  Research & Development   
3   33        No  Travel_Frequently       1392  Research & Development   
4   27        No      Travel_Rarely        591  Research & Development   

   distancefromhome  education educationfield  employeenumber  \
0                 1          2  Life Sciences               1   
1                 8          1  Life Sciences               2   
2                 2          2          Other               4   
3                 3          4  Life Sciences               5   
4                 2          1        Medical               7   

   environmentsatisfaction  gender  hourlyrate  jobinvolvement  joblevel  \
0                        2  Female          94               3  